# Neural Networks - From Theory to Implementation

In this notebook we cover:

1. What neural networks are
2. How they work internally
3. How to use them with scikit-learn
4. Data preparation and optimization
5. Implementation from scratch (NumPy)

Goal:
Understand neural networks as systems, not black boxes.

## What is a Neural Network?


### Problem

Simple models (linear classifiers) cannot model complex patterns.

Example:
    class = 1 if (x1 * x2 > 0)

This is nonlinear --> cannot be solved with a line.


### Manual Solution

Define many rules manually.

Problem:
- does not scale
- becomes complex quickly


### Neural Network Idea

Learn complex patterns automatically:

    input --> transformations --> output


### Key Concept

A neural network is:

    repeated matrix multiplication + nonlinear functions


### Structure

Input --> Hidden Layer(s) --> Output

Each layer transforms data into a new representation.


## How Neural Networks Work

Each layer does:

1. Linear transformation:

    Z = XW + b

2. Nonlinear activation:

    A = activation(Z)


### Why activation is critical

Without it:

    linear + linear + linear = still linear

--> useless for complex patterns


### With activation

 nonlinear transformations
 complex decision boundaries


### Training

Neural networks learn by:

1. Predict
2. Measure error (loss)
3. Compute gradients
4. Update parameters

## How to implement this using scikit
(I will show u how the code of an actuall nn looks, but lets do this later!)

In [ ]:
# DATA PREPARATION 

# WHAT: load and prepare data
# WHY: neural networks require clean numerical input

import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

data = load_iris()

X = data.data
y = data.target

# scaling is important for neural networks
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## Why Scaling Matters

Neural networks use gradient descent.

If features have different scales:

--> gradients become unstable  
--> training becomes inefficient  

Scaling ensures:

- uniform contribution of features
- stable optimization

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [ ]:
# SCIKIT NEURAL NETWORK 

# WHAT: ready-made neural network
# WHY: abstraction over complex math

from sklearn.neural_network import MLPClassifier

model = MLPClassifier(
    hidden_layer_sizes=(50,),   # one hidden layer with 50 neurons, there are many more parameter u can read into!
    max_iter=1000
)

model.fit(X_train, y_train)


In [ ]:
from sklearn.metrics import accuracy_score

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))

## Optimization in Neural Networks

Neural networks use:

    Gradient Descent + Backpropagation


### Goal

Minimize loss:

    L(y, ŷ)


### Process

1. Forward pass → prediction  
2. Compute loss  
3. Backward pass (gradients)  
4. Update weights  


### Important Parameters

- learning rate --> step size  
- epochs --> iterations  
- layers --> model complexity  



### Key Insight

Optimization = adjusting weights to reduce error

## From Scratch
(Try to understand as much as possible! This is the real working of a neural network. I think its important that u have seen this once and neural networks are the most simple version of ML. Probably noone is ever going to ask u to build one urself especially not the more advanced ones we r gonna learn in the future)

In [ ]:
# SIMPLE CUSTOM DATA 

np.random.seed(42)

X = np.random.randn(200, 2)
y = (X[:,0] * X[:,1] > 0).astype(int)
y = y.reshape(-1, 1)

In [ ]:
# initialize parameters

W1 = np.random.randn(2, 4) * 0.1
b1 = np.zeros((1, 4))

W2 = np.random.randn(4, 1) * 0.1
b2 = np.zeros((1, 1))

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

In [ ]:
def forward(X):
    Z1 = X @ W1 + b1
    A1 = np.tanh(Z1)

    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)

    return Z1, A1, Z2, A2

In [ ]:
def compute_loss(y, y_hat):
    eps = 1e-8
    return -np.mean(y * np.log(y_hat + eps) +
                    (1 - y) * np.log(1 - y_hat + eps))


In [ ]:
def backward(X, y, Z1, A1, Z2, A2):
    m = X.shape[0]

    dZ2 = A2 - y
    dW2 = A1.T @ dZ2 / m
    db2 = np.mean(dZ2, axis=0)

    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * (1 - np.tanh(Z1)**2)

    dW1 = X.T @ dZ1 / m
    db1 = np.mean(dZ1, axis=0)

    return dW1, db1, dW2, db2

In [ ]:
import matplotlib.pyplot as plt

# DECISION BOUNDARY
def plot_boundary(step):
    xx, yy = np.meshgrid(
        np.linspace(X[:,0].min(), X[:,0].max(), 100),
        np.linspace(X[:,1].min(), X[:,1].max(), 100)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    
    _, _, _, probs = forward(grid)
    Z = probs.reshape(xx.shape)

    plt.contourf(xx, yy, Z > 0.5, alpha=0.3)
    plt.scatter(X[:,0], X[:,1], c=y.flatten())
    plt.title(f"Decision Boundary at step {step}")
    plt.show()


In [ ]:
learning_rate = 0.1
losses = []

for epoch in range(1000):

    Z1, A1, Z2, A2 = forward(X)

    loss = compute_loss(y, A2)
    losses.append(loss)

    dW1, db1, dW2, db2 = backward(X, y, Z1, A1, Z2, A2)

    # update
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    # show boundary every 200 steps
    if epoch % 200 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")
        plot_boundary(epoch)

plt.plot(losses)
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

## What You Just Built

We implemented:

- forward pass  
- loss computation  
- gradient calculation  
- parameter updates  


### 1. Forward Pass

The forward pass computes the prediction:

    input --> hidden layer --> output

This is just:

    matrix multiplication + activation functions



### 2. Loss Computation

We defined a loss function:

    Cross-Entropy Loss

It measures:

- how wrong the prediction is  
- how confident the model is  

Important:

- correct + confident --> low loss  
- wrong + confident --> high loss  


### 3. Gradient Calculation

We computed gradients using backpropagation.

This answers:

    How much does each parameter contribute to the error?

Gradients point in the direction of increasing error.


### 4. Parameter Updates (Optimization)

We update parameters using gradient descent:

    parameter = parameter - learning_rate × gradient

This means:

--> move parameters in the direction that reduces loss  


### Key Insight

All neural networks are:

    matrix multiplications + nonlinearities + gradients

Training =

    repeatedly reducing loss by updating parameters


### Important Difference

scikit-learn:

    abstraction

- hides forward pass  
- hides gradient computation  
- hides optimization  

your implementation:

    explicit mechanics

- you see how predictions are computed  
- you see how loss is defined  
- you control how learning happens  


### Final Takeaway

A neural network is:

    a function that is gradually improved
    by minimizing a loss function

Optimization shapes the decision boundary over time.
